In [0]:
data=[(1, "Mawa", "Data Engineer", 85000),
    (2, "Rahul", "Data Analyst", 65000),
    (3, "Priya", "Data Scientist", 95000),
    (4, "Suresh", "Azure Admin", 60000)]
columns = ["emp_id", "emp_name", "role", "salary"]

# 3. Create DataFrame
df = spark.createDataFrame(data, schema=columns)

# 4. Display DataFrame
df.show()

In [0]:
df_select= df.select("emp_name", 'role','emp_id');
df_select.show()

In [0]:
from pyspark.sql.functions import col
##df_high_salary = df.filter(df.salary>70000)
df_high_salary = df.filter(col("salary")>70000)

df_high_salary.show()

In [0]:
df_withcol = df.withColumn("bonus",col("salary")*0.1)
df_withcol.show()

In [0]:
df_renamed =df.withColumnRenamed("role","designation")
df_renamed.show()

In [0]:
from pyspark.sql.functions import avg, count
df_grouped = df.groupBy("role").agg(
    avg("salary").alias("avg_salary"),
    count("emp_id").alias("total_emp")
)
df_grouped.show()


In [0]:
from pyspark.sql.functions import col
df_select1 = df.filter((col("salary")>65000)).withColumn("bonus_salary",col("salary")*0.1).select("emp_name","bonus_salary")
#withColumn("bonus_salary",col("salary")*0.1)
df_select1.show()

In [0]:
# 1. Dummy CSV Content రాయడం
#csv_data = """flight_id,origin,destination,delay
#F101,HYD,BLR,15
#F102,MAA,DEL,0
#F103,BLR,BOM,45
#F104,DEL,HYD,10"""

# 2. ఫైల్‌ని DBFS లో సేవ్ చేయడం
#dbutils.fs.put("/tmp/my_flights.csv", csv_data, overwrite=True)
#with open("/tmp/my_flights.csv", "w") as f:
  #  f.write(csv_data)

# 3. spark.read తో రీడ్ చేయడం
#df_csv = spark.read \
  #  .format("csv") \
  #  .option("header", "true") \
  #  .option("inferSchema", "true") \
  #  .load("/tmp/my_flights.csv")

#df_csv.show()
#df_csv.printSchema()

# Unity Catalog default table read చేయడం
df_nyctaxi = spark.read.table("samples.nyctaxi.trips")

# top 5 rows
df_nyctaxi.show(5)

# Schema డిజైన్ చూడటం
df_nyctaxi.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Custom Schema
schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("delay", IntegerType(), True)
])

# 2. Sample Data
data = [
    ("F101", "HYD", "BLR", 15),
    ("F102", "MAA", "DEL", 0),
    ("F103", "BLR", "BOM", 45)
]

# 3. Create DataFrame with explicit schema
df_sample = spark.createDataFrame(data, schema=schema)
df_sample.show()
df_sample.printSchema()

In [0]:
# 1. Credentials Setup
storage_account_name = "adlsmawa123"  # నీ Storage Account Name ఇక్కడ ఇవ్వు
storage_account_key = dbutils.secrets.get(scope="<your-secret-scope>", key="storage-account-key")  # Databricks Secret Scope నుండి Key తీసుకోవడం
container_name = "rawdata"

# 2. Spark Session లో Storage Account కీ ని సెట్ చేయడం
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
# ABFSS Path Construction
file_name = "airlines.csv"  # నువ్వు అప్‌లోడ్ చేసిన CSV ఫైల్ పేరు (eg: flights.csv)
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"

# spark.read తో నేరుగా Azure Lake నుండి రీడ్ చేయడం
df_adls = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(adls_path)

# Display Data & Schema
df_adls.show(10)
df_adls.printSchema()

In [0]:

#store spark.read to adls gen2 (stoarge account)
# 1. డేటాని ఫిల్టర్ చేయడం (ఉదాహరణకి 2003 ఇయర్ డేటా మాత్రమే)
# నీ కాలమ్ నేమ్‌తో మ్యాచ్ చేస్కో
df_filtered = df_adls.filter(df_adls["`Time.Year`"] == 2003)

# 2. Output Path (ADLS Gen2 లోపల processed-data అనే ఫోల్డర్)
output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/processed_airports"

# 3. Parquet ఫార్మాట్‌ లోకి సేవ్ చేయడం
df_filtered.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_path)

print("Data written successfully to ADLS Gen2! 🚀")

# Parquet ఫైల్ ని రీడ్ చేయడం
df_parquet_check = spark.read.parquet(output_path)
df_parquet_check.show(5)

In [0]:
from pyspark.sql.functions import col, sum, avg, count, max, min

# 1. Airport.Code ద్వారా గ్రూప్ చేసి Total Delays & Avg Delays లెక్కించడం
df_grouped = df_adls.groupBy("`Airport.Code`").agg(
    sum("`Statistics.# of Delays.Carrier`").alias("total_carrier_delays"),
    avg("`Statistics.# of Delays.Carrier`").alias("avg_carrier_delays"),
    count("`Time.Label`").alias("total_records")
)

# 2. Delays ఎక్కువగా ఉన్న ఎయిర్‌పోర్ట్స్‌ని సరిగ్గా డిస్ప్లే చేయడానికి Order By (Sort)
df_grouped_sorted = df_grouped.orderBy(col("total_carrier_delays").desc())

df_grouped_sorted.show(5)

In [0]:
# 1. Airport Details కోసం ఒక శాంపిల్ Look-up DataFrame తయారుచేద్దాం
airport_details_data = [
    ("ATL", "Hartsfield-Jackson", "Large Hub"),
    ("BOS", "Boston Logan", "Medium Hub"),
    ("BWI", "Baltimore/Washington", "Medium Hub"),
    ("HYD", "Rajiv Gandhi Intl", "International") # ద్యాని డేటా మన df_adls లో లేదు
]

schema_details = ["Airport_Code", "Full_Name", "Airport_Category"]

df_airport_details = spark.createDataFrame(airport_details_data, schema=schema_details)
df_airport_details.show()

In [0]:
#Inner join
df_inner = df_adls.join(
    df_airport_details,
    df_adls["`Airport.Code`"] == df_airport_details["Airport_Code"],
    how="inner"
)
# కాలమ్స్ లోని చుక్కలను (_) అండర్‌స్కోర్‌గా మార్చడం
df_clean = df_adls
for column in df_clean.columns:
    new_col = column.replace(".", "_").replace(" ", "_").replace("#", "num")
    df_clean = df_clean.withColumnRenamed(column, new_col)

# ఇప్పుడు చూడండి కాలమ్ పేర్లు ఎంత క్లీన్ అయిపోయాయో!
df_clean.printSchema()

df_inner.select("Airport_Code", "Full_Name", "Airport_Category", "`Time.Year`").show(5)

In [0]:
#left join
df_left = df_adls.join(
    df_airport_details,
    df_adls["`Airport.Code`"] == df_airport_details["Airport_Code"],
    how="left"
)
# కాలమ్స్ లోని చుక్కలను (_) అండర్‌స్కోర్‌గా మార్చడం
df_clean = df_adls
for column in df_clean.columns:
    new_col = column.replace(".", "_").replace(" ", "_").replace("#", "num")
    df_clean = df_clean.withColumnRenamed(column, new_col)

# ఇప్పుడు చూడండి కాలమ్ పేర్లు ఎంత క్లీన్ అయిపోయాయో!
df_clean.printSchema()


df_left.select("`Airport.Code`", "Full_Name", "Airport_Category").show(5)

In [0]:
#Right join
df_right = df_adls.join(
    df_airport_details,
    df_adls["`Airport.Code`"] == df_airport_details["Airport_Code"],
    how="right"
)

df_right.select("Airport_Code", "Full_Name", "`Time.Year`").show(5)

In [0]:
from pyspark.sql.functions import current_timestamp, col, sum, avg, count

# -------------------------------------------------------------------------
# 0. PRE-CLEANING: Delta ఫార్మాట్ కోసం కాలమ్ నేమ్స్ లో ఉన్న Spaces/Dots తీసేయడం
# -------------------------------------------------------------------------
df_clean_source = df_adls
for column in df_clean_source.columns:
    new_col = column.replace(".", "_").replace(" ", "_").replace("#", "num")
    df_clean_source = df_clean_source.withColumnRenamed(column, new_col)


# -------------------------------------------------------------------------
# 1. BRONZE LAYER: Clean Schema + Ingestion Timestamp + Delta Write
# -------------------------------------------------------------------------
df_bronze = df_clean_source.withColumn("ingestion_time", current_timestamp())

bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/bronze_airports"

df_bronze.write.format("delta").mode("overwrite").save(bronze_path)
print("🥉 Bronze Layer Completed!")


# -------------------------------------------------------------------------
# 2. SILVER LAYER: Read Bronze + Deduplication/Validation
# -------------------------------------------------------------------------
df_silver = spark.read.format("delta").load(bronze_path).dropDuplicates()

silver_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/silver_airports"

df_silver.write.format("delta").mode("overwrite").save(silver_path)
print("🥈 Silver Layer Completed!")


# -------------------------------------------------------------------------
# 3. GOLD LAYER: Read Silver + Business Aggregations
# -------------------------------------------------------------------------
df_silver_read = spark.read.format("delta").load(silver_path)

df_gold = df_silver_read.groupBy("Airport_Code").agg(
    sum("Statistics_num_of_Delays_Carrier").alias("total_carrier_delays"),
    avg("Statistics_num_of_Delays_Carrier").alias("avg_carrier_delays"),
    count("Time_Label").alias("total_records")
)

gold_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/gold_airports"

df_gold.write.format("delta").mode("overwrite").save(gold_path)
print("🥇 Gold Layer Completed Successfully!")

In [0]:
# Gold Layer లో సేవ్ అయిన Delta Table ని రీడ్ చేయడం
gold_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/gold_airports"

df_gold_final = spark.read.format("delta").load(gold_path)

# Results Display
df_gold_final.orderBy(col("total_carrier_delays").desc()).show(10)

In [0]:
# 1. Gold Table యొక్క చరిత్ర (History) చూడటం
gold_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/gold_airports"

display(spark.sql(f"DESCRIBE HISTORY delta.`{gold_path}`"))

In [0]:
# Version 0 లో ఉన్న డేటాని రీడ్ చేయడం (Time Travel)
df_version_0 = spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load(gold_path)

df_version_0.show(5)

In [0]:
# Source Data: ATL కి కొత్త Delays కౌంట్ (Update) + HYD అని కొత్త ఎయిర్‌పోర్ట్ (Insert)
new_data = [
    ("ATL", 99999, 500.0, 100),  # Existing code -> Should UPDATE
    ("HYD", 1200, 15.0, 10)       # New code -> Should INSERT
]

schema = ["Airport_Code", "total_carrier_delays", "avg_carrier_delays", "total_records"]
df_updates = spark.createDataFrame(new_data, schema=schema)

# దీన్ని తాత్కాలికంగా Temp View గా మారుద్దాం
df_updates.createOrReplaceTempView("staged_updates")

In [0]:
# Gold Delta Path ని SQL క్వెరీలో వాడడం
spark.sql(f"""
    MERGE INTO delta.`{gold_path}` AS target
    USING staged_updates AS source
    ON target.Airport_Code = source.Airport_Code
    WHEN MATCHED THEN
      UPDATE SET 
        target.total_carrier_delays = source.total_carrier_delays,
        target.avg_carrier_delays = source.avg_carrier_delays,
        target.total_records = source.total_records
    WHEN NOT MATCHED THEN
      INSERT (Airport_Code, total_carrier_delays, avg_carrier_delays, total_records)
      VALUES (source.Airport_Code, source.total_carrier_delays, source.avg_carrier_delays, source.total_records)
""")

print("✅ MERGE INTO Completed Successfully!")

In [0]:
# MERGE అయ్యాక ATL అప్‌డేట్ అయిందో, HYD యాడ్ అయిందో చూద్దాం
df_gold_updated = spark.read.format("delta").load(gold_path)
df_gold_updated.filter(col("Airport_Code").isin("ATL", "HYD")).show()

In [0]:
# 1. Database/Schema create cheyadam
spark.sql("CREATE DATABASE IF NOT EXISTS medallion_db")

# 2. ADLS Gold path ni SQL External Table ga register cheyadam
#gold_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/gold_airports"

#spark.sql(f"""
 #   CREATE TABLE IF NOT EXISTS medallion_db.gold_airports
  #  LOCATION '{gold_path}'
#""")

#print("✅ SQL Table registered successfully!")
#--another way---
# 1. Database Create చేయడo
#spark.sql("CREATE DATABASE IF NOT EXISTS medallion_db")

# 2. DataFrame ని డెల్టా టేబుల్‌గా డెటాబ్రిక్స్ SQL కి రిజిస్టర్ చేయడం
df_gold_updated.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion_db.gold_airports")

print("✅ Managed SQL Table registered successfully!")

In [0]:
%sql
SELECT * FROM medallion_db.gold_airports ORDER BY total_carrier_delays DESC;

In [0]:
%sql
-- 1. Table schema & metadata chudadam
DESCRIBE EXTENDED medallion_db.gold_airports;

In [0]:
%sql
---1.aggregations & top delays check chey:
-- 2. Direct Business Query
SELECT 
    Airport_Code,
    total_carrier_delays,
    avg_carrier_delays
FROM medallion_db.gold_airports
WHERE total_carrier_delays > 1000
ORDER BY total_carrier_delays DESC;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, rank, dense_rank, lag, lead

# 1. Silver Layer నుంచి డేటా రీడ్ చేద్దాం
silver_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/medallion/silver_airports"
df_silver = spark.read.format("delta").load(silver_path)

# 2. Window Spec నిర్వచించడం (Time_Year ఆధారంగా పార్టిషన్ చేసి, Delays బట్టి సర్చడం)
window_spec = Window.partitionBy("Time_Year").orderBy(col("Statistics_num_of_Delays_Carrier").desc())

# 3. Row Number, Rank, Dense Rank అప్లై చేయడం
df_ranked = df_silver.withColumn("row_num", row_number().over(window_spec)) \
                     .withColumn("rank", rank().over(window_spec)) \
                     .withColumn("dense_rank", dense_rank().over(window_spec))

df_ranked.select("Time_Year", "Airport_Code", "Statistics_num_of_Delays_Carrier", "row_num", "rank", "dense_rank") \
         .filter(col("row_num") <= 3) \
         .show(10)

# 4. Lag & Lead తో ప్రతీ ఇయర్ డిలేలను గత ఇయర్ డిలేలతో పోల్చడం
window_lead_lag = Window.partitionBy("Airport_Code").orderBy("Time_Year")

df_lag_lead = df_silver.withColumn("prev_year_delays", lag("Statistics_num_of_Delays_Carrier", 1).over(window_lead_lag)) \
                       .withColumn("next_year_delays", lead("Statistics_num_of_Delays_Carrier", 1).over(window_lead_lag))

df_lag_lead.select("Airport_Code", "Time_Year", "Statistics_num_of_Delays_Carrier", "prev_year_delays", "next_year_delays") \
           .filter(col("Airport_Code") == "ATL") \
           .show(10)